In [1]:
# Import objects from pyomo package
from pyomo.environ import ConcreteModel, SolverFactory, TransformationFactory, value, units

# Import the main FlowsheetBlock from IDAES. The flowsheet block will contain the unit model
from idaes.core import FlowsheetBlock

# Import idaes logger to set output levels
import idaes.logger as idaeslog

# Import the IAPWS property package to create a properties block for the flowsheet
from idaes.models.properties import iapws95
from idaes.models.properties.helmholtz.helmholtz import PhaseType

# Import the mixer unit model from IDAES
from idaes.models.unit_models import Mixer, MomentumMixingType

# Importing feed streams for the mixer unit model
from idaes.models.unit_models import Feed, Product

# Import the degrees of freedom function to check that the model is well posed
from idaes.core.util.model_statistics import degrees_of_freedom 
    
# Import the Arc and TransformationFactory from pyomo.network
from pyomo.network import Arc 

# Importing diagnostic tools to check that the model is well initialized
from idaes.core.util.model_diagnostics import DiagnosticsToolbox

# Importing the propogate state
from idaes.core.util.initialization import propagate_state

In [2]:
# Create the ConcreteModel and the FlowsheetBlock, and attach the flowsheet block to it.
m = ConcreteModel()

m.fs = FlowsheetBlock(
    dynamic=False
)  # dynamic or ss flowsheet needs to be specified here

# Add properties parameter block to the flowsheet with specifications
m.fs.properties = iapws95.Iapws95ParameterBlock(
    phase_presentation=PhaseType.LG,
    state_vars = iapws95.StateVars.PH)

# Creating an instance of mixing unit model and attaching it to the flowsheet
m.fs.mixer = Mixer(
    property_package=m.fs.properties,
    num_inlets=2,
    momentum_mixing_type=MomentumMixingType.minimize,
)

# Creating feed streams for the mixer unit model and attaching them to the flowsheet
m.fs.feed1 = Feed(property_package=m.fs.properties)
m.fs.feed2 = Feed(property_package=m.fs.properties)

# Creating product stream for the mixer unit model and attaching it to the flowsheet
m.fs.product = Product(property_package=m.fs.properties)


In [3]:
# Connecting the feed and product streams to the mixer unit model
m.fs.s01 = Arc(source=m.fs.feed1.outlet, destination=m.fs.mixer.inlet_1)
m.fs.s02 = Arc(source=m.fs.feed2.outlet, destination=m.fs.mixer.inlet_2)
m.fs.s03 = Arc(source=m.fs.mixer.outlet, destination=m.fs.product.inlet)

# Transforming the arcs to create the necessary constraints for the model
TransformationFactory("network.expand_arcs").apply_to(m)

# Calling the degree of freedom function to check that the model is well posed
print("The degree of freedom for the model is : {}".format(degrees_of_freedom(m)))

The degree of freedom for the model is : 6


In [4]:
# Fixing the feed conditions for the mixer unit model using Pressure and Enthalpy
# Stream 1: 300 K, 101325 Pa, 1 mol/s
enth_1 = iapws95.htpx(T=350 * units.K, P=101325 * units.Pa)  # Calculate enthalpy
m.fs.feed1.properties[0].flow_mol.fix(55.5084)
m.fs.feed1.properties[0].enth_mol.fix(enth_1)
m.fs.feed1.properties[0].pressure.fix(101325)

# Stream 2: 350 K, 101325 Pa, 1 mol/s
enth_2 = iapws95.htpx(T=450 * units.K, P=501325 * units.Pa)  # Calculate enthalpy
m.fs.feed2.properties[0].flow_mol.fix(55.5084)
m.fs.feed2.properties[0].enth_mol.fix(enth_2)
m.fs.feed2.properties[0].pressure.fix(501325)

# Calling the degree of freedom function again to check that the model is well posed after fixing the feed conditions
print("The degree of freedom for the model is : {}".format(degrees_of_freedom(m)))

The degree of freedom for the model is : 0


In [5]:
# Initializing the model using the default initialization routine for the mixer unit model
m.fs.feed1.initialize(outlvl=idaeslog.INFO, state_args={})
m.fs.feed2.initialize(outlvl=idaeslog.INFO, state_args={})

# Propogating the state from the feed streams to the mixer unit model
propagate_state(arc=m.fs.s01, direction="forward") 
propagate_state(arc=m.fs.s02, direction="forward") 

# Initializing the mixer unit model using the default initialization routine for the mixer unit model
m.fs.mixer.initialize(outlvl=idaeslog.DEBUG)

# Propogating the state from the mixer unit model to the product stream
propagate_state(arc=m.fs.s03, direction="forward")

2026-04-05 06:01:39 [INFO] idaes.init.fs.feed1: Initialization Complete.
2026-04-05 06:01:39 [INFO] idaes.init.fs.feed2: Initialization Complete.
2026-04-05 06:01:40 [DEBUG] idaes.solve.fs.mixer: Ipopt 3.13.2: nlp_scaling_method=gradient-based
2026-04-05 06:01:40 [DEBUG] idaes.solve.fs.mixer: tol=1e-06
2026-04-05 06:01:40 [DEBUG] idaes.solve.fs.mixer: max_iter=200
2026-04-05 06:01:40 [DEBUG] idaes.solve.fs.mixer: 
2026-04-05 06:01:40 [DEBUG] idaes.solve.fs.mixer: 
2026-04-05 06:01:40 [DEBUG] idaes.solve.fs.mixer: ******************************************************************************
2026-04-05 06:01:40 [DEBUG] idaes.solve.fs.mixer: This program contains Ipopt, a library for large-scale nonlinear optimization.
2026-04-05 06:01:40 [DEBUG] idaes.solve.fs.mixer:  Ipopt is released as open source code under the Eclipse Public License (EPL).
2026-04-05 06:01:40 [DEBUG] idaes.solve.fs.mixer:          For more information visit http://projects.coin-or.org/Ipopt
2026-04-05 06:01:40 [DEB

In [6]:
# Creating an instance of the DiagnosticsToolbox to check that the model is well initialized
dt = DiagnosticsToolbox(m)

# Printing any structual issues with the model
print(dt.report_structural_issues())

# Printing any numerical issues with the model
print(dt.report_numerical_issues())

# Printing large residuals in the model
print(dt.display_constraints_with_large_residuals())

# Wrapping and Printing any infeasibilities with the model
try:
    print(dt.compute_infeasibility_explanation())
except Exception as e:
    print("Model is feasible - no infeasibility explanation needed.")

# Printing variables inside or outside the bounds
print(dt.display_variables_at_or_outside_bounds())

Model Statistics

        Activated Blocks: 19 (Deactivated: 0)
        Free Variables in Activated Constraints: 14 (External: 0)
            Free Variables with only lower bounds: 0
            Free Variables with only upper bounds: 0
            Free Variables with upper and lower bounds: 8
        Fixed Variables in Activated Constraints: 6 (External: 0)
        Activated Equality Constraints: 14 (Deactivated: 0)
        Activated Inequality Constraints: 0 (Deactivated: 0)
        Activated Objectives: 0 (Deactivated: 0)

------------------------------------------------------------------------------------
0 WARNINGS

    No warnings found!

------------------------------------------------------------------------------------
0 Cautions

    No cautions found!

------------------------------------------------------------------------------------
Suggested next steps:

    Try to initialize/solve your model and then call report_numerical_issues()

None
Model Statistics

    Jacobian Con

In [7]:
# Solving the model using the ipopt solver
solver = SolverFactory("ipopt")
solver_status = solver.solve(m, tee=True)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

In [8]:
# Displaying the results
print(m.fs.feed1.report())
print(m.fs.feed2.report())
print(m.fs.mixer.report())



Unit : fs.feed1                                                            Time: 0.0
------------------------------------------------------------------------------------
    Stream Table
                         Units          Outlet  
    Molar Flow          mole / second     55.508
    Mass Flow       kilogram / second     1.0000
    T                          kelvin     350.00
    P                          pascal 1.0132e+05
    Vapor Fraction      dimensionless     0.0000
    Molar Enthalpy       joule / mole     5798.0
None

Unit : fs.feed2                                                            Time: 0.0
------------------------------------------------------------------------------------
    Stream Table
                         Units          Outlet  
    Molar Flow          mole / second     55.508
    Mass Flow       kilogram / second     1.0000
    T                          kelvin     450.00
    P                          pascal 5.0132e+05
    Vapor Fraction      dimensi

In [9]:
m.fs.visualize("Mixer Model")

2026-04-05 06:01:42 [INFO] idaes.idaes_ui.fv.fsvis: Started visualization server
2026-04-05 06:01:42 [INFO] idaes.idaes_ui.fv.fsvis: Loading saved flowsheet from 'Mixer Model.json'
2026-04-05 06:01:42 [INFO] idaes.idaes_ui.fv.fsvis: Saving flowsheet to default file 'Mixer Model.json' in current directory (/workspace/08 IDAES PSE Models/00 Mixer Modelling)
2026-04-05 06:01:42 [WARNING] idaes.idaes_ui.fv.fsvis: Flowsheet name changed: old='Mixer Model' new='Mixer-Model'
Flowsheet name changed to 'Mixer-Model'
2026-04-05 06:01:42 [INFO] idaes.idaes_ui.fv.fsvis: Flowsheet visualization at: http://localhost:50891/app?id=Mixer-Model


VisualizeResult(store=<idaes_ui.fv.persist.FileDataStore object at 0x7baa8c32abc0>, port=50891, server=<idaes_ui.fv.model_server.FlowsheetServer object at 0x7baa8c816260>, save_diagram=<bound method SaveDiagramScreenshot.save_diagram_screenshot of <idaes_ui.fv.save_diagram_screenshot.SaveDiagramScreenshot object at 0x7baa8c816ce0>>)